In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

# === 1. Indstil parametre ===
mappe_path = "/Users/alhu/Data/Performance"          # <-- SKIFT denne til din mappe
benchmark_fil = "/Users/alhu/Data/Performance/Benchmark_10.xlsx"

In [ ]:
benchmark_path = os.path.join(mappe_path, benchmark_fil)

# === 2. Indlæs benchmark-data ===
benchmark_df = pd.read_excel(benchmark_path)

# === 3. Find alle andre Excel-filer i mappen ===
excel_files = [f for f in os.listdir(mappe_path)
               if f.endswith('.xlsx') and f != benchmark_fil]

# === 4. Initialiser resultatlister ===
results = []

# === 5. Opsæt sentence-transformers-model ===
model = SentenceTransformer('all-MiniLM-L6-v2')

for filename in excel_files:
    fil_path = os.path.join(mappe_path, filename)
    try:
        df = pd.read_excel(fil_path)
        # --- LABEL ---
        label_match = (df['label'] == benchmark_df['label'])
        accuracy = label_match.mean()
        # --- SCORE ---
        score_diff = abs(df['score'] - benchmark_df['score'])
        mean_abs_error = score_diff.mean()
        # --- BEGRUNDELSE / TEXT ---
        emb1 = model.encode(df['begrunde'].astype(str).tolist(), show_progress_bar=False)
        emb2 = model.encode(benchmark_df['begrunde'].astype(str).tolist(), show_progress_bar=False)
        similarities = [util.cos_sim(a, b).item() for a, b in zip(emb1, emb2)]
        avg_similarity = sum(similarities) / len(similarities)
        # --- Gem resultat ---
        results.append({
            "fil": filename,
            "label_accuracy": accuracy,
            "score_mae": mean_abs_error,
            "text_similarity": avg_similarity
        })
    except Exception as e:
        print(f"Fejl ved {filename}: {e}")

# === 6. Lav DataFrame af resultater ===
results_df = pd.DataFrame(results)


# === 7. Plot resultater ===
plt.figure(figsize=(14,5))
plt.subplot(1,3,1)
plt.bar(results_df['fil'], results_df['label_accuracy'])
plt.ylabel('Label Accuracy')
plt.ylim(0,1)
plt.title('Label Accuracy')
plt.xticks(rotation=45, ha='right')

plt.subplot(1,3,2)
plt.bar(results_df['fil'], results_df['score_mae'])
plt.ylabel('Mean Absolute Error')
plt.title('Score-afvigelse')
plt.xticks(rotation=45, ha='right')

plt.subplot(1,3,3)
plt.bar(results_df['fil'], results_df['text_similarity'])
plt.ylabel('Tekst-similarity')
plt.ylim(0,1)
plt.title('Tekstlighed')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

# --- BONUS: Vis resultater som tabel ---
results_df[['fil','label_accuracy','score_mae','text_similarity']]